In [1]:
# dictionary = {'parameter_name': [[min, max], default]}
CORES = 8

class SparkParameters():
    def __init__(self):
        self.boolean_parameters = {'spark.broadcast.compress': [['true', 'false'], 'true'],
                                  'spark.memory.offHeap.enabled': [['true', 'false'], 'true'],
                                  'spark.rdd.compress': [['true', 'false'], 'true'],
                                  'spark.shuffle.compress': [['true', 'false'], 'true'],
                                  'spark.shuffle.spill.compress': [['true', 'false'], 'true'],
                                  'spark.sql.codegen.aggregate.map.twolevel.enable': [['true', 'false'], 'true'],
                                  'spark.sql.inMemoryColumnarStorage.compressed': [['true', 'false'], 'true'],
                                  'spark.sql.inMemoryColumnarStorage.partitionPruning': [['true', 'false'], 'true'],
                                  'spark.sql.join.preferSortMergeJoin': [['true', 'false'], 'true'],
                                  'spark.sql.retainGroupColumns': [['true', 'false'], 'true'],
                                  'spark.sql.sort.enableRadixSort': [['true', 'false'], 'true']
                                  }

        self.unit_mb_parameters = {'spark.broadcast.blockSize': [[1, 16], 4],
                                   'spark.driver.memory': [[1024, 4096], 1024],
                                   'spark.executor.memory': [[1024, 4096], 1024], # memory + memoryOverhead >= server memory
                                   'spark.executor.memoryOverhead': [[0, 8192], 384], # memory + memoryOverhead >= server memory
                                   'spark.kryoserializer.buffer.max': [[8, 128], 64],
                                   'spark.memory.offHeap.size': [[0, 49152], 0],
                                   'spark.reducer.maxSizeInFlight': [[24, 144], 48],
                                   'spark.shuffle.accurateBlockThreshold': [[100, 1000], 100],
                                   'spark.shuffle.service.index.cache.size': [[10, 2048], 100],
                                   'spark.storage.memoryMapThreshold': [[1, 10], 1]
                                   }

        self.unit_kb_parameters = {'spark.io.compression.zstd.bufferSize': [[16, 96], 32],
                                   'spark.kryoserializer.buffer': [[2, 128], 64],
                                   'spark.shuffle.file.buffer': [[16, 96], 32],
                                   'spark.sql.autoBroadcastJoinThreshold': [[1024, 8192], 1024]
                                   }

        self.numerical_parameters = {'spark.default.parallelism':[[CORES, 1000], CORES],
                                     'spark.driver.cores': [[1, CORES], 1],
                                     'spark.executor.cores': [[1, CORES], 3],
                                     'spark.executor.instances': [[2, 112], 2],
                                     'spark.io.compression.zstd.level': [[1, 5], 1],
                                     'spark.locality.wait': [[1, 6], 3],
                                     'spark.scheduler.revive.interval': [[1, 5], 1],
                                     'spark.shuffle.io.numConnectionsPerPeer': [[1, 5], 1],
                                     'spark.shuffle.sort.bypassMergeThreshold': [[100, 1000], 200],
                                     'spark.speculation.interval': [[10, 1000], 100],
                                     'spark.sql.cartesianProductExec.buffer.in.memory.threshold': [[1024, 8192], 4096],
                                     'spark.sql.codegen.maxFields': [[50, 200], 100],
                                     'spark.sql.inMemoryColumnarStorage.batchSize': [[5000, 20000], 10000],
                                     'spark.sql.shuffle.partitions': [[100, 1000], 200]
                                     }

        self.continous_parameters = {'spark.memory.fraction': [[0.5, 0.9], 0.6],
                                     'spark.memory.storageFraction': [[0.5, 0.9], 0.5]
                                     }
        
        self._get_combined_parameters()
        self._get_min_max_array()
        self._get_configuration_template()
        
        self.len_boolean = len(self.boolean_parameters)
        self.len_unit_kb = len(self.unit_kb_parameters)
        self.len_unit_mb = len(self.unit_mb_parameters)
        self.len_numerical = len(self.numerical_parameters)
        self.len_continuous = len(self.continous_parameters)
        
        self.parameter_names = list(self.all_parameters.keys())
        
    def __len__(self):
        return len(self.all_parameters)
    
    def _get_combined_parameters(self):
        self.all_parameters = self.boolean_parameters.copy()
        self.all_parameters.update(self.unit_kb_parameters)
        self.all_parameters.update(self.unit_mb_parameters)
        self.all_parameters.update(self.numerical_parameters)
        self.all_parameters.update(self.continous_parameters)
        
    def _get_min_max_array(self):
        self.ub = list()
        self.lb = list()
        for k, v in self.all_parameters.items():
            if k in self.boolean_parameters:
                self.lb.append(0)
                self.ub.append(1)
            else:
                self.lb.append(v[0][0])
                self.ub.append(v[0][1])
    
    def _get_configuration_template(self):
        f = open('spark.conf.template', 'r')
        self.conf_template = f.readlines()
        f.close()
    
    def save_configuration_file(self, values):
        conf_file = open('spark.conf', 'w')
        conf_file.writelines(self.conf_template)
        print('$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$')
        for i, p in enumerate(self.parameter_names):
            if p in self.boolean_parameters:
                v = 'true' if round(values[i])==1 else 'false'
            elif p in self.continous_parameters:
                v = round(values[i], 1)
            elif p in self.unit_kb_parameters:
                v = str(round(values[i]))+'k'
            elif p in self.unit_mb_parameters:
                v = str(round(values[i]))+'m'
            else:
                v = round(values[i])
            
            conf_file.writelines(f'{p}={v}\n')
            print(f'{p}={v}')
        print('$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$')
        conf_file.close()

In [2]:
sp = SparkParameters()

In [3]:
import os
import pandas as pd
import numpy as np

import baxus
from baxus.util.behaviors import BaxusBehavior
from baxus.util.behaviors.gp_configuration import GPBehaviour
from baxus.baxus import BAxUS
from baxus.util.parsing import embedding_type_mapper, acquisition_function_mapper, mle_optimization_mapper
from baxus.benchmarks.benchmark_function import Benchmark

/home/jieun/anaconda3/envs/py3.8/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
embedding_type = "baxus" # "hesbo"
acquisition_function = "ts" # "ei"
mle_optimization = "sample-and-choose-best" # "multistart-gd"

In [5]:
bin_sizing_method = embedding_type_mapper[embedding_type]
acquisition_function = acquisition_function_mapper[acquisition_function]
mle_optimization_method = mle_optimization_mapper[mle_optimization]

In [6]:
target_dim = 10 # args.target_dim
n_init = 10 # args.n_init # initial samples (pre-observed samples)
max_evals = 100 # args.max_evals
noise_std = 0 # args.noise_std
new_bins_on_split = None # args.new_bins_on_split 
multistart_samples = 100 # args.multistart_samples
mle_training_steps = 50 # args.mle_training_steps
multistart_after_samples = 10 # args.multistart_after_sample
l_init = 0.8 # args.initial_baselength
l_min = 0.5**7 # args.min_baselength
l_max = 1.6 # args.max_baselength
adjust_initial_target_dim = True # args.adjust_initial_target_dimension
budget_until_input_dim = 0 # args.budget_until_input_dim

In [7]:
behavior = BaxusBehavior(n_new_bins=2,#new_bins_on_split,
                         initial_base_length=l_init,
                         min_base_length=l_min,
                         max_base_length=l_max,
                         acquisition_function=acquisition_function,
                         embedding_type=bin_sizing_method,
                         adjust_initial_target_dim=adjust_initial_target_dim,
                         noise=noise_std,
                         budget_until_input_dim=budget_until_input_dim
                        )

In [8]:
gp_behaviour = GPBehaviour(
    mll_estimation=mle_optimization_method,
    n_initial_samples=multistart_samples,
    n_best_on_lhs_selection=multistart_after_samples,
    n_mle_training_steps=mle_training_steps,
)

In [33]:
class SparkBench(Benchmark):
    def __init__(self, dim, ub, lb, sp, bench_type=None):
        super().__init__(dim=dim, ub=ub, lb=lb, noise_std=0)
        self.sp = sp # spark parameter infos
        self.bench_type = bench_type
    
    def __call__(self, x):
        
        self.sp.save_configuration_file(x)
        self.execute_spark_bench()
        self.res, _ = self.get_results()
        
        return res
        
    def execute_spark_bench(self):
        # transport a generated configuration to the master server
        os.system('sshpass scp spark.conf jieun@34.64.230.62:/home/jieun/HiBench/conf')
        os.system('sshpass ssh jieun@34.64.230.62 /home/jieun/scripts/run_terasort.sh')
    
    def get_results(self):
        f = open('hibench.report', 'r')
        report = f.readlines()
        f.close()
        duration = report[-1].split()[-3]
        tps = report[-1].split()[-2]
        return duration, tps
#         if x.ndim == 0:
#             x = np.expand_dims(x, 0)
#         if x.ndim == 1:
#             x = np.expand_dims(x, 0)
#         assert x.ndim == 2
#         global cnt
#         cnt += 1
#         print(f"##########{cnt}th regression run##########")
#         res = regr.predict(x)
# #         res = res.squeeze()
# #         return -(res[0]/res[1])
#         return -(res[:, 0]/res[:,1])

In [34]:
input_dim = len(sp) # args.input_dim
ub = sp.ub
lb = sp.lb

f = SparkBench(dim=input_dim, ub=ub, lb=lb, sp=sp)

optim = BAxUS(f=f,
              n_init=n_init,
              max_evals=max_evals,
              target_dim=target_dim,
              behavior=behavior,
              gp_behaviour=gp_behaviour
              )

In [35]:
optim.optimize()

$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
spark.broadcast.compress=true
spark.memory.offHeap.enabled=true
spark.rdd.compress=false
spark.shuffle.compress=false
spark.shuffle.spill.compress=false
spark.sql.codegen.aggregate.map.twolevel.enable=false
spark.sql.inMemoryColumnarStorage.compressed=false
spark.sql.inMemoryColumnarStorage.partitionPruning=true
spark.sql.join.preferSortMergeJoin=true
spark.sql.retainGroupColumns=true
spark.sql.sort.enableRadixSort=true
spark.io.compression.zstd.bufferSize=78k
spark.kryoserializer.buffer=100k
spark.shuffle.file.buffer=78k
spark.sql.autoBroadcastJoinThreshold=2644k
spark.broadcast.blockSize=4m
spark.driver.memory=1718m
spark.executor.memory=1718m
spark.executor.memoryOverhead=7004m
spark.kryoserializer.buffer.max=35m
spark.memory.offHeap.size=42021m
spark.reducer.maxSizeInFlight=41m
spark.shuffle.accurateBlockThreshold=231m
spark.shuffle.service.index.cache.size=1587m
spark.storage.memoryMapThreshold=9m
spark.default.parallelism=232
spark.driver.cores=7


Parsing conf: /home/jieun/HiBench/conf/hadoop.conf
Parsing conf: /home/jieun/HiBench/conf/hibench.conf
Parsing conf: /home/jieun/HiBench/conf/spark.conf
Parsing conf: /home/jieun/HiBench/conf/workloads/micro/terasort.conf
probe sleep jar: /usr/lib/hadoop-mapreduce/hadoop-mapreduce-client-jobclient-2.10.2-tests.jar
hdfs rm -r: /usr/lib/hadoop/bin/hadoop --config /usr/lib/hadoop/etc/hadoop fs -rm -r -skipTrash hdfs://cluster-test-m/HiBench/Terasort/Input


Deleted hdfs://cluster-test-m/HiBench/Terasort/Input
Submit MapReduce Job: /usr/lib/hadoop/bin/hadoop --config /usr/lib/hadoop/etc/hadoop jar /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar teragen -D mapreduce.job.maps=8 -D mapreduce.job.reduces=8 32000000 hdfs://cluster-test-m/HiBench/Terasort/Input


23/10/11 13:09:11 INFO client.RMProxy: Connecting to ResourceManager at cluster-test-m/10.178.0.21:8032
23/10/11 13:09:11 INFO client.AHSProxy: Connecting to Application History server at cluster-test-m/10.178.0.21:10200
23/10/11 13:09:11 INFO terasort.TeraGen: Generating 32000000 using 8
23/10/11 13:09:11 INFO mapreduce.JobSubmitter: number of splits:8
23/10/11 13:09:12 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1696995318686_0034
23/10/11 13:09:12 INFO conf.Configuration: resource-types.xml not found
23/10/11 13:09:12 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
23/10/11 13:09:12 INFO resource.ResourceUtils: Adding resource type - name = memory-mb, units = Mi, type = COUNTABLE
23/10/11 13:09:12 INFO resource.ResourceUtils: Adding resource type - name = vcores, units = , type = COUNTABLE
23/10/11 13:09:12 INFO impl.YarnClientImpl: Submitted application application_1696995318686_0034
23/10/11 13:09:12 INFO mapreduce.Job: The url to track the job: h

finish HadoopPrepareTerasort bench
patching args=
start ScalaSparkTerasort bench


...

23/10/11 13:09:41 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
23/10/11 13:09:41 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
23/10/11 13:09:41 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
23/10/11 13:09:41 INFO org.spark_project.jetty.util.log: Logging initialized @2316ms to org.spark_project.jetty.util.log.Slf4jLog
23/10/11 13:09:41 INFO org.spark_project.jetty.server.Server: jetty-9.4.z-SNAPSHOT; built: unknown; git: unknown; jvm 1.8.0_382-b05
23/10/11 13:09:41 INFO org.spark_project.jetty.server.Server: Started @2405ms
23/10/11 13:09:41 INFO org.spark_project.jetty.server.AbstractConnector: Started ServerConnector@1df98368{HTTP/1.1, (http/1.1)}{0.0.0.0:4040}
23/10/11 13:09:42 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at cluster-test-m/10.178.0.21:8032
23/10/11 13:09:42 INFO org.apache.hadoop.yarn.client.AHSProxy: Connecting to Application History server at cluster-test-m/10.178.0.21:10200
23/10

Spent 40ms computing base-splits.
Spent 3ms computing TeraScheduler splits.


23/10/11 13:10:47 INFO org.spark_project.jetty.server.AbstractConnector: Stopped Spark@1df98368{HTTP/1.1, (http/1.1)}{0.0.0.0:4040}


finish ScalaSparkTerasort bench
68.341


AssertionError: 